# Notebook 03 — Logistic Regression from Scratch

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 03 of 15  
**Time estimate:** 90 minutes

> Logistic regression is the most important ML model for biological classification.
> It's interpretable, probabilistic, and the direct foundation of neural networks.
> You must be able to derive it, implement it, and explain it without notes.

---
## Step 1 — Motivation

Cancer vs. normal, binding vs. non-binding, pathogenic vs. benign — biological
classification problems are everywhere. Logistic regression is the workhorse:
interpretable coefficients, probabilistic output (not just a hard label), and direct
extension to multi-class via softmax. Understanding it from scratch is prerequisite
for understanding deep learning.

---
## Step 2 — Intuition

**Why not linear regression for classification?**
Linear regression predicts $\hat{y} \in (-\infty, \infty)$ but probabilities must be
in $[0,1]$. For binary outcomes, linear regression's predictions go out of range.

**The sigmoid function** maps any real number to $(0,1)$:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Model: $P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b)$.

**Decision boundary:** $P(y=1) = 0.5$ when $\mathbf{w}^T \mathbf{x} + b = 0$.
This is a hyperplane — logistic regression is a **linear classifier** in input space.

**Coefficient interpretation:**
$w_j$ is the change in **log-odds** per unit increase in feature $j$:
$$\log \frac{P(y=1)}{P(y=0)} = \mathbf{w}^T \mathbf{x} + b$$

$e^{w_j}$ is the **odds ratio** — a 1-unit increase in feature $j$ multiplies the
odds of class 1 by $e^{w_j}$.

---
## Step 3 — Biological Background

**Cancer classification from gene expression:**
- Features: expression levels of selected genes
- Target: cancer subtype (binary or multi-class)
- Coefficients: which genes have the strongest association with each subtype

**Variant pathogenicity:**
- Features: conservation scores, MAF, functional annotations
- Target: pathogenic (1) vs. benign (0)
- Key challenge: extreme class imbalance (most variants are benign)

**Logistic regression assumptions:**
1. Log-odds are linear in features
2. Observations are independent
3. Little multicollinearity (correlated features cause unstable coefficients)
4. Large enough sample size for reliable MLE estimates

**Separation problem:** If the classes are perfectly linearly separable, MLE pushes
coefficients to $\pm\infty$ ("complete separation"). Regularization (L2) prevents this.
Common in high-dimensional genomics — always use regularization.

---
## Step 4 — Mathematical Explanation

**Model:** $\hat{p}_i = \sigma(\mathbf{w}^T \mathbf{x}_i)$ where $\mathbf{x}_i$ includes a bias term.

**Likelihood** (Bernoulli):
$$\mathcal{L}(\mathbf{w}) = \prod_{i=1}^n \hat{p}_i^{y_i} (1 - \hat{p}_i)^{1-y_i}$$

**Log-likelihood / cross-entropy loss** (negated for minimization):
$$\mathcal{J}(\mathbf{w}) = -\frac{1}{n}\sum_{i=1}^n \left[ y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i) \right]$$

**Gradient:**
$$\nabla_\mathbf{w} \mathcal{J} = \frac{1}{n} X^T (\hat{\mathbf{p}} - \mathbf{y})$$

Same form as linear regression gradient! The difference is that $\hat{\mathbf{p}} = \sigma(X\mathbf{w})$
is nonlinear — making the loss non-quadratic (no closed form, need iterative optimization).

**With L2 regularization (ridge):**
$$\mathcal{J}(\mathbf{w}) = -\frac{1}{n}\sum_i [y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)] + \frac{\lambda}{2}\|\mathbf{w}\|^2$$
$$\nabla_\mathbf{w} \mathcal{J} = \frac{1}{n} X^T(\hat{\mathbf{p}} - \mathbf{y}) + \lambda \mathbf{w}$$

**Multi-class (Softmax regression):**
$$P(y=k | \mathbf{x}) = \frac{e^{\mathbf{w}_k^T \mathbf{x}}}{\sum_{j=1}^K e^{\mathbf{w}_j^T \mathbf{x}}}$$

Loss: categorical cross-entropy $\mathcal{J} = -\frac{1}{n}\sum_i \log P(y=y_i | \mathbf{x}_i)$.

In [ ]:
# Step 6 — Python: Logistic regression from scratch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ---- Utilities ----
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def cross_entropy_loss(y, p_hat, eps=1e-10):
    p_hat = np.clip(p_hat, eps, 1 - eps)
    return -np.mean(y * np.log(p_hat) + (1-y) * np.log(1 - p_hat))

# ---- Logistic regression with gradient descent ----
class LogisticRegressionScratch:
    def __init__(self, lr=0.1, n_iter=1000, l2=0.01):
        self.lr = lr
        self.n_iter = n_iter
        self.l2 = l2
        self.w = None
        self.losses = []
    
    def fit(self, X, y):
        n, p = X.shape
        self.w = np.zeros(p + 1)  # includes bias
        Xb = np.column_stack([np.ones(n), X])
        for i in range(self.n_iter):
            z = Xb @ self.w
            p_hat = sigmoid(z)
            # Gradient (bias term not regularized: w[0])
            grad = Xb.T @ (p_hat - y) / n
            reg = self.l2 * self.w
            reg[0] = 0  # don't regularize bias
            self.w -= self.lr * (grad + reg)
            self.losses.append(cross_entropy_loss(y, p_hat))
        return self
    
    def predict_proba(self, X):
        Xb = np.column_stack([np.ones(len(X)), X])
        return sigmoid(Xb @ self.w)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# ---- Generate synthetic cancer classification data ----
rng = np.random.default_rng(42)

# 2 features (gene expression), 200 samples, ~balanced
def make_cancer_data(n=200, seed=42):
    rng = np.random.default_rng(seed)
    # Cancer: gene A high, gene B low
    X_cancer = rng.multivariate_normal([2, -1], [[0.8, 0.3], [0.3, 0.8]], n//2)
    # Normal: gene A low, gene B high
    X_normal = rng.multivariate_normal([-1, 2], [[0.8, -0.2], [-0.2, 0.8]], n//2)
    X = np.vstack([X_cancer, X_normal])
    y = np.array([1]*(n//2) + [0]*(n//2))
    return X, y

X, y = make_cancer_data(200)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Fit from scratch
lr_scratch = LogisticRegressionScratch(lr=0.5, n_iter=500, l2=0.01)
lr_scratch.fit(X_train_sc, y_train)

# Fit sklearn (for comparison)
lr_sk = LogisticRegression(C=100, random_state=42)  # C=1/lambda
lr_sk.fit(X_train_sc, y_train)

acc_scratch = (lr_scratch.predict(X_test_sc) == y_test).mean()
acc_sk = lr_sk.score(X_test_sc, y_test)
print(f'Test accuracy — Scratch: {acc_scratch:.4f}, sklearn: {acc_sk:.4f}')
print(f'Coefficients (scratch): bias={lr_scratch.w[0]:.4f}, w={lr_scratch.w[1:].round(4)}')
print(f'Coefficients (sklearn): bias={lr_sk.intercept_[0]:.4f}, w={lr_sk.coef_[0].round(4)}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Decision boundary + data
ax = axes[0]
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
Z = lr_scratch.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6, vmin=0, vmax=1)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)
ax.scatter(X_train_sc[y_train==1, 0], X_train_sc[y_train==1, 1],
             c='tomato', s=20, alpha=0.7, label='Cancer (train)')
ax.scatter(X_train_sc[y_train==0, 0], X_train_sc[y_train==0, 1],
             c='steelblue', s=20, alpha=0.7, label='Normal (train)')
ax.scatter(X_test_sc[y_test==1, 0], X_test_sc[y_test==1, 1],
             c='tomato', s=60, marker='*', zorder=3, label='Cancer (test)')
ax.scatter(X_test_sc[y_test==0, 0], X_test_sc[y_test==0, 1],
             c='steelblue', s=60, marker='*', zorder=3, label='Normal (test)')
ax.set_xlabel('Gene A (standardized)')
ax.set_ylabel('Gene B (standardized)')
ax.set_title('A. Logistic regression\ndecision boundary')
ax.legend(fontsize=6)

# Panel B: Training loss convergence
ax = axes[1]
ax.plot(lr_scratch.losses, 'steelblue', lw=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('B. Training loss (gradient descent)')
ax.axhline(lr_scratch.losses[-1], color='red', ls='--', lw=0.8,
             label=f'Final={lr_scratch.losses[-1]:.4f}')
ax.legend(fontsize=8)

# Panel C: Predicted probability distribution by class
ax = axes[2]
probs = lr_scratch.predict_proba(X_test_sc)
ax.hist(probs[y_test==0], bins=15, alpha=0.6, color='steelblue', label='Normal', density=True)
ax.hist(probs[y_test==1], bins=15, alpha=0.6, color='tomato', label='Cancer', density=True)
ax.axvline(0.5, color='black', ls='--', lw=1.5, label='Decision threshold')
ax.set_xlabel('Predicted P(cancer)')
ax.set_ylabel('Density')
ax.set_title('C. Predicted probabilities by class\n(good separation → reliable classifier)')
ax.legend(fontsize=8)

plt.suptitle('Module 13 NB03: Logistic Regression from Scratch', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Derive the gradient $\nabla_\mathbf{w} \mathcal{J}$ for logistic regression by hand.
   Note the similarity to the linear regression gradient.
2. Add L2 regularization to `LogisticRegressionScratch`. Verify that for large $\lambda$,
   the coefficients shrink toward 0.
3. Implement the softmax function and categorical cross-entropy loss for multi-class
   logistic regression. Test on a 3-class problem.
4. The sigmoid has a maximum gradient of 0.25 at $z=0$. What does this imply for
   gradient flow in deep networks? (Preview: vanishing gradient problem.)

---
## Step 10 — Quiz

1. Why can't you use linear regression for classification?
2. Write the cross-entropy loss formula. Why do we maximize log-likelihood rather than
   likelihood directly?
3. What is the decision boundary of logistic regression? What type of classifier is it?
4. What is the interpretation of coefficient $w_j = 0.5$ in a logistic regression model?
5. What is the separation problem, and how does L2 regularization prevent it?

---
## Step 12 — Reflection

> *[In one paragraph: explain why logistic regression coefficients are interpretable
> in terms of odds ratios. In a cancer classification model, what would it mean if
> gene X has coefficient $w_X = 1.8$?]*

---
*Next: `04_decision_trees_and_random_forests.ipynb`*